In [ ]:
# 7회차 — 연기 positive를 조리 규모로 재조정 (v1, 2026-08-04)
print('round7 notebook v1')
!nvidia-smi -L
!pip -q install ultralytics==8.3.* kagglehub

In [ ]:
# [1] 업로드 — kitchen-fire-poc.zip · assets_1~6.zip
import zipfile, os, glob
from google.colab import files
up = files.upload()
os.makedirs('/content/work', exist_ok=True)
for n in up:
    zipfile.ZipFile(n).extractall('/content/work')
os.chdir('/content/work')
for d in ('bases','flamelib','smokelib','smokelib_thin','negsrc',
          'eval_neg','eval_steam','eval_cooksmoke','weights'):
    p = f'assets/{d}'
    print(f'{d:16s}', len(glob.glob(p+'/*')) if os.path.isdir(p) else '없음')

In [ ]:
# [2] D-Fire
import kagglehub, os
DFIRE = kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo')
print(DFIRE)

In [ ]:
# [3] 평가셋 — A-fire·C·학습배경, A-smoke
!python scripts/dfire_eval_set.py --dfire "$DFIRE" --out eval
print('-'*60)
!python scripts/dfire_smoke_eval.py --dfire "$DFIRE" --out eval

In [ ]:
# [4] 합성 — 얇은 소재 라이브러리 + 조리 규모 연기 70%
!python scripts/synthesize_smoke.py --assets assets --out ds7 \
    --dfire-bg-list eval/train_bg.txt --dfire-bg-count 600 --haze-prob 0.5 \
    --thin-prob 0.7 --smokelib smokelib_thin --seed 20260805
!cat ds7/data.yaml

In [ ]:
# [4-1] 합성 확인 — 연기 장면 12장
import cv2, glob, numpy as np, random
from google.colab.patches import cv2_imshow
fs = [f for f in sorted(glob.glob('ds7/images/train/*_smoke.jpg')) if 'fire_smoke' not in f]
random.seed(0); sel = random.sample(fs, 12); tiles = []
for f in sel:
    im = cv2.imread(f); H, W = im.shape[:2]
    for line in open(f.replace('/images/', '/labels/').replace('.jpg', '.txt')):
        c, x, y, bw, bh = line.split(); x, y, bw, bh = map(float, (x, y, bw, bh))
        cv2.rectangle(im, (int((x-bw/2)*W), int((y-bh/2)*H)),
                          (int((x+bw/2)*W), int((y+bh/2)*H)), (255, 180, 0), 2)
    tiles.append(cv2.resize(im, (400, 225)))
cv2_imshow(np.vstack([np.hstack(tiles[i:i+4]) for i in range(0, 12, 4)]))

In [ ]:
# [5] 학습 (약 70분)
!yolo detect train model=yolov8s.pt data=ds7/data.yaml epochs=60 imgsz=640 \
    batch=16 project=/content/runs name=r7 exist_ok=False

In [ ]:
# [6] 채점 — 5그룹 + 화염 회귀 (6회차 대비)
import glob, os
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
print('가중치:', best)
!python scripts/eval_gate6.py --weights "$best" --eval-dir eval \
    --cctv assets/eval_neg --steam assets/eval_steam --conf 0.10

In [ ]:
# [7] 주 지표 — 조리 연기(E) vs 수증기(D) 판별비
import glob, os, math, numpy as np
from ultralytics import YOLO
best = max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime)
m = YOLO(best); FIRE, SMOKE, CONF = 0, 1, 0.10
E = sorted(glob.glob('assets/eval_cooksmoke/*.jpg'))
S = sorted(glob.glob('assets/eval_steam/*.jpg'))

def scan(paths, b=32):
    out = []
    for i in range(0, len(paths), b):
        for r in m.predict(paths[i:i+b], conf=0.03, verbose=False):
            f = s = 0.0
            if len(r.boxes):
                cl = r.boxes.cls.cpu().numpy().astype(int)
                cf = r.boxes.conf.cpu().numpy()
                if (cl == FIRE).any(): f = float(cf[cl == FIRE].max())
                if (cl == SMOKE).any(): s = float(cf[cl == SMOKE].max())
            out.append((f, s))
    return np.array(out)

zE, zS = scan(E), scan(S)
print(f'E {len(E)}장 · D {len(S)}장')
for t in (0.03, 0.05, 0.10, 0.15, 0.25, 0.40):
    e = (zE[:, SMOKE] >= t).mean(); s = (zS[:, SMOKE] >= t).mean()
    r = e/s if s else float('nan')
    print(f'conf {t:4.2f} | E {e*100:5.1f}%  D {s*100:5.1f}%  판별비 {r:5.2f}'
          + ('  <- 운용' if abs(t-CONF) < 1e-9 else ''))
e = (zE[:, SMOKE] >= CONF).mean(); s = (zS[:, SMOKE] >= CONF).mean()
r = e/s if s else float('nan')
v = '성립' if r >= 1.0 else ('개선했으나 미달' if r >= 0.7 else '실패')
print(f'\n판별비 {r:.2f} (6회차 0.54) -> 사전 등록 판정: {v}')
import collections
d = collections.defaultdict(lambda: [0, 0])
for p, z in zip(E, zE):
    v2 = os.path.basename(p).split('_')[0]; d[v2][1] += 1
    if z[SMOKE] >= CONF: d[v2][0] += 1
for k in sorted(d):
    a, b2 = d[k]; print(f'{k:14s} {a:3d}/{b2:3d} = {a/b2*100:5.1f}%')

In [ ]:
# [8] 가중치 내려받기
import glob, os
from google.colab import files
files.download(max(glob.glob('/content/runs/*/weights/best.pt'), key=os.path.getmtime))